# Angular Steering evaluation visualization


In [ ]:
from pathlib import Path
from configs import MAX_SIM_DIR_ID, MAX_NORM_DIR_ID

dir_id = "max_sim"

if dir_id == "max_sim":
    DIR_ID_MAP = MAX_SIM_DIR_ID
elif dir_id == "max_norm":
    DIR_ID_MAP = MAX_NORM_DIR_ID

visualization_dir = Path(f"visualization/{dir_id}/")
visualization_dir.mkdir(parents=True, exist_ok=True)


## Refusal score and Harmful scores


In [ ]:
import json
import numpy as np
import plotly
from pathlib import Path
from plotly.subplots import make_subplots
import plotly.graph_objects as go

model_ids = [
    "Qwen/Qwen2.5-3B-Instruct",
    "Qwen/Qwen2.5-7B-Instruct",
    "Qwen/Qwen2.5-14B-Instruct",
    "meta-llama/Llama-3.2-3B-Instruct",
    "meta-llama/Llama-3.1-8B-Instruct",
    "google/gemma-2-9b-it",
]

language = "en"
data_type = "harmful"
num_row = 3
num_col = len(model_ids) // num_row
data_path = "output/"

adaptive = True

fig = make_subplots(
    rows=num_row,
    cols=num_col,
    specs=[[{"type": "polar"}] * num_col] * num_row,
    subplot_titles=[model_id.split("/")[1] for model_id in model_ids],
    vertical_spacing=0.01,
    horizontal_spacing=0.1,
)


colour_map = {
    "substring_matching": plotly.colors.qualitative.Plotly[1],
    "llamaguard3": plotly.colors.qualitative.Plotly[2],
    "harmbench": plotly.colors.qualitative.Plotly[0],
}

method_colour_map = {
    "PID": plotly.colors.qualitative.Plotly[0],
    "DIM": plotly.colors.qualitative.Plotly[1],
    "PI" : plotly.colors.qualitative.Plotly[2],
    "PD" : plotly.colors.qualitative.Plotly[3],
    "RePE": plotly.colors.qualitative.Plotly[4],
    "ITI":plotly.colors.qualitative.Plotly[5],
}

categories = list(str(i) for i in range(0, 360, 10))
categories.append(categories[0])

for idx, model_id in enumerate(model_ids):
    _, model_name = model_id.split("/")
    output_path = Path(data_path) / model_name

    if adaptive:
        glob_pattern = f"*eval-mode_1-llamaguard3*.json"
    else:
        glob_pattern = "*eval-[!(mode)(perp)]*.json"

    for file in sorted(list(output_path.glob(glob_pattern))):
        if "PID_" in file.stem.split("-")[0]:
            method = "PID"
        elif "PI_" in file.stem.split("-")[0] :
            method = "PI"
        elif "PD_" in  file.stem.split("-")[0] :
            method = "PD" 
        elif "RePE_" in file.stem.split("-")[0] :
            method = "RePE"
        elif "ITI_" in file.stem.split("-")[0] :
            method = "ITI"
        else:
            method = "DIM"
        
        print(method)
        if adaptive:
            metric = file.stem.split("-")[2]
        else:
            metric = file.stem.split("-")[1]
        print(metric)
        if metric == "llmjudge":
            continue
        if metric not in colour_map:
            colour_map[metric] = plotly.colors.qualitative.Plotly[len(colour_map)]

        with open(file, "r") as f:
            eval_data = json.load(f)
        print(file)
        baseline = eval_data["baseline"]
        if isinstance(baseline, list):
            baseline = np.mean(baseline)
        fig.add_trace(
            go.Scatterpolar(
                r=[
                    (
                        1 - baseline
                        if metric in ["substring_matching", "llmjudge"]
                        else baseline
                    )
                    for _ in range(len(categories))
                ],
                theta=categories,
                name="baseline",
                line=dict(width=2, color=colour_map[metric], dash="dot"),
                # legendgroup="legend"
                mode="lines",
                opacity=0.5,
                showlegend=False,
            ),
            row=idx // num_col + 1,
            col=idx % num_col + 1,
        )

        print(eval_data.keys())
        chosen_plane_id = [
            s
            for s in eval_data.keys()
            if "dir_random" not in s and "baseline" not in s
        ]
        print(chosen_plane_id)
        if not chosen_plane_id:
            continue
        print("hehe")
        print(model_id, chosen_plane_id)
        chosen_plane_id = chosen_plane_id[0]
        values = [eval_data[chosen_plane_id][cat] for cat in categories]

        # if metric == "llmjudge":
        #     values = [[0.5 if v == 0.75 else v for v in val] for val in values]
        values = [np.mean(val) if isinstance(val, list) else val for val in values]
        values.append(values[0])
        fig.add_trace(
            go.Scatterpolar(
                r=(
                    [1 - v for v in values]
                    if metric in ["substring_matching", "llmjudge"]
                    else values
                ),
                theta=categories,
                legend=f"legend{idx if idx > 0 else ''}",
                # fill="toself",
                name=method,
                line=dict(width=2, color=method_colour_map[method]),
                # legendgroup="legend"
                mode="lines",
                # opacity=0.5,
                showlegend=idx == 0,
            ),
            row=idx // num_col + 1,
            col=idx % num_col + 1,
        )

    fig.add_trace(
        go.Scatterpolar(
            r=[1.02],
            # theta=["0"],
            name="feature direction",
            marker=dict(size=20, symbol="arrow-right", color="black"),
            mode="markers",
            showlegend=idx == 0,
        ),
        row=idx // num_col + 1,
        col=idx % num_col + 1,
    )

    # fig.add_trace(
    #     go.Scatterpolar(
    #         r=[1],
    #         theta=['90'],
    #         name="1st PC direction",
    #         marker=dict(size=10, symbol=2, color=plotly.colors.qualitative.Plotly[len(colour_map) + 1]),
    #         mode="markers",
    #         showlegend=idx == 0,
    #     ),
    #     row=idx // num_col + 1,
    #     col=idx % num_col + 1,
    # )

for i in range(len(model_ids) + 1):
    polar_key = f'polar{i if i > 0 else ""}'
    fig.update_layout(
        {
            polar_key: dict(
                radialaxis=dict(
                    visible=True,
                    dtick=0.2,
                    tickfont=dict(size=20),
                ),
                angularaxis=dict(
                    # direction="clockwise",
                    # rotation=0,  # 0 degrees at the right (East)
                    # period=360,
                    # tickmode="array",
                    tickvals=list(
                        range(0, 360, 10)
                    ),  # Show all degrees in 10 intervals
                    ticktext=[
                        f"{i}°" for i in range(0, 360, 10)
                    ],  # Show all tick labels
                    tickfont=dict(size=18),  # Smaller font to fit all labels
                    dtick=10,
                ),
            )
        }
    )


# Global layout settings
fig.update_layout(
    height=2000,
    width=1200,
    # title_text="Angular Steering Effects on Model Performance",
    showlegend=True,
    legend=dict(
        orientation="h",
        # yanchor="top",
        y=0.0,
        xanchor="center",
        x=0.5,
        # entrywidth=0,
        font=dict(size=30),
    ),
    margin=dict(l=50, r=50, t=20, b=0),
)
fig.update_annotations(font=dict(size=36), yshift=-20)

fig.show()

if adaptive:
    output_name = f"eval_adaptive-harmness-all_models-vertical"
else:
    output_name = f"eval-harmness-all_models-vertical"

fig.write_image(
    visualization_dir / f"PID_{output_name}.pdf",
    width=1200,
    height=2000,
    scale=5,
)

In [ ]:
# !plotly_get_chrome

## LLM-as-a-judge classification results


In [ ]:
model_ids = [
    "Qwen/Qwen2.5-3B-Instruct",
    "Qwen/Qwen2.5-7B-Instruct",
    "Qwen/Qwen2.5-14B-Instruct",
    "meta-llama/Llama-3.2-3B-Instruct",
    "meta-llama/Llama-3.1-8B-Instruct",
    "google/gemma-2-9b-it",
]

language = "en"
data_type = "harmful"
num_row = 1
num_col = len(model_ids) // num_row
data_path = "output/"
adaptive = True


colour_map = {
    "direct": "red",
    "indirect": "orange",
    "redirect": "teal",
    "refusal": "green",
}
score2label = {
    0: "refusal",
    0.25: "redirect",
    0.5: "indirect",
    0.75: "indirect",
    1: "direct",
}

fig = make_subplots(
    rows=num_row,
    cols=num_col,
    specs=[[{"type": "polar"}] * num_col] * num_row,
    subplot_titles=[model_id.split("/")[1] for model_id in model_ids],
    vertical_spacing=0.01,
    horizontal_spacing=0.1,
)

for idx, model_id in enumerate(model_ids):
    _, model_name = model_id.split("/")
    output_path = Path(data_path) / model_name

    if adaptive:
        file = list(output_path.glob("*eval-mode_1-llmjudge-*.json"))[0]
    else:
        file = list(output_path.glob("*eval-llmjudge-*.json"))[0]
    print(file)
    with open(file, "r") as f:
        eval_data = json.load(f)

    chosen_plane_id = [s for s in eval_data.keys() if DIR_ID_MAP[model_id] in s and "dir_random" not in s]
    if len(chosen_plane_id) != 1:
        print(f"Skipping {model_id} due to {len(chosen_plane_id)} planes IDs found: {chosen_plane_id}")
        continue

    chosen_plane_id = chosen_plane_id[0]


    print(model_id, chosen_plane_id)
    categories = list(str(i) for i in range(0, 360, 10))
    categories.append(categories[0])
    values = [eval_data[chosen_plane_id][cat] for cat in categories]

    scores = [1, 0.75, 0.25, 0]
    for v in scores:
        r = [vals.count(v) / len(vals) for vals in values]
        fig.add_trace(
            go.Barpolar(
                r=r,
                # theta=list(str(i) for i in range(0, 360, 10)),
                # marker_color=[colour_map[score2label[v]] for v in scores],
                marker_color=colour_map[score2label[v]],
                name=score2label[v],
                showlegend=idx == 0,
                marker=dict(
                    line=dict(
                        width=0.0,
                        # color="rgba(0,0,0,0)"
                    )
                ),
            ),
            row=idx // num_col + 1,
            col=idx % num_col + 1,
        )

    fig.add_trace(
        go.Scatterpolar(
            r=[1.02],
            theta=["0"],
            name="feature direction",
            marker=dict(size=20, symbol="arrow-right", color="black"),
            mode="markers",
            showlegend=idx == 0,
        ),
        row=idx // num_col + 1,
        col=idx % num_col + 1,
    )


polar_config = dict(
    bargap=0,
    hole=0,
    angularaxis=dict(tickfont=dict(size=16), dtick=10),
    radialaxis=dict(
        showticklabels=False,
        showline=False,
    ),
)
fig.update_layout(
    polar=polar_config,
    polar2=polar_config,
    polar3=polar_config,
    polar4=polar_config,
    polar5=polar_config,
    polar6=polar_config,
    showlegend=True,
    legend=dict(
        orientation="h",
        # yanchor="top",
        y=0.0,
        xanchor="center",
        x=0.5,
        font=dict(size=30),
    ),
    height=2000,
    width=1200,
    margin=dict(l=50, r=50, t=20, b=0),
)

fig.update_annotations(font=dict(size=36), yshift=-20)

fig.show()

fig.write_image(
    visualization_dir / "eval_adaptive-llmjudge-all_models-vertical.pdf",
    width=1200,
    height=2000,
    scale=5,
)

# Adaptive Angular Steering on tinyBenchmark


In [ ]:
import json
import os
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import re
from glob import glob
from pathlib import Path
from collections import defaultdict

# Base directory for benchmarks
base_dir = f"benchmarks_old/{dir_id}/"
# base_dir = f"benchmarks_old/{dir_id}/"

# Dictionary to map tasks to their primary metric(s)
task_to_metrics = {
    "tinyArc": ["acc_norm,none"],
    "tinyGSM8k": [
        "exact_match,strict-match",
        "exact_match,flexible-extract",
    ],
    "tinyHellaswag": ["acc_norm,none"],
    "tinyMMLU": ["acc_norm,none"],
    "tinyTruthfulQA": ["acc,none"],
    "tinyWinogrande": ["acc_norm,none"],
}


models = [
    "Qwen2.5-3B-Instruct",
    "Qwen2.5-7B-Instruct",
    "Qwen2.5-14B-Instruct",
    "Llama-3.2-3B-Instruct",
    "Llama-3.1-8B-Instruct",
    "gemma-2-9b-it",
]

# Collect data for all models and tasks
results = defaultdict(lambda: defaultdict(dict))
baselines = defaultdict(dict)

# Debug tracking
found_metrics = defaultdict(set)

for task_dir in os.listdir(base_dir):
    task_path = os.path.join(base_dir, task_dir)
    task_path = os.path.join(task_path, "")
    if not os.path.isdir(task_path):
        continue

    for model_dir in os.listdir(task_path):
        if model_dir not in models:
            continue

        model_path = os.path.join(task_path, model_dir)
        if not os.path.isdir(model_path):
            continue

        # Get metrics for this task
        metrics = task_to_metrics.get(task_dir, [])
        if not metrics:
            continue

        # Find all results files for this model and task
        for degree in list(range(0, 360, 10)) + ["none"]:
            pattern = os.path.join(
                model_path, f"adaptive_{degree}", "*", "results_*.json"
            )
            result_files = glob(pattern)

            if not result_files:
                continue

            # Since the result files are named by date, sort them to get the latest
            result_fiile = sorted(result_files)
            # Use the last result file found for each degree
            result_file = result_files[-1]

            try:
                with open(result_file, "r") as f:
                    data = json.load(f)

                # For each metric defined for this task
                for metric in metrics:
                    # Create a display name for the metric
                    if task_dir == "tinyGSM8k":
                        # For GSM8k, include the metric type in the display name
                        # Extract strict-match or loose-match
                        metric_type = (metric.split(",")[1]).split("-")[0]
                        display_name = f"{task_dir} ({metric_type})"
                    else:
                        display_name = task_dir

                    if (
                        task_dir in data.get("results", {})
                        and metric in data["results"][task_dir]
                    ):
                        score = data["results"][task_dir][metric]
                        found_metrics[task_dir].add(
                            metric
                        )  # Track which metrics were found

                        if degree == "none":
                            baselines[model_dir][display_name] = score
                        else:
                            results[model_dir][display_name][int(degree)] = score
            except Exception as e:
                print(f"Error processing {result_file}: {e}")

# Print metrics found for debugging
print("Found metrics by task:")
for task, metrics in found_metrics.items():
    print(f"{task}: {metrics}")

# Create subplots - arrange based on number of models
n_models = len(models)
n_cols = min(2, n_models)
n_rows = (n_models + n_cols - 1) // n_cols
# n_cols = 1
# n_rows = n_models

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    specs=[[{"type": "polar"} for _ in range(n_cols)] for _ in range(n_rows)],
    subplot_titles=models,
    vertical_spacing=0.01,
    horizontal_spacing=0.1,
)

# Add reference baseline trace only once (for legend)
fig.add_trace(
    go.Scatterpolar(
        r=[0],
        theta=[0],
        name="baseline",
        line=dict(width=2, color="black", dash="dot"),
        mode="lines",
        opacity=0.5,
        showlegend=True,
    ),
    row=1,
    col=1,
)

# Use Plotly's built-in color palette
colors = px.colors.qualitative.Plotly

# Plot data for each model
for model_idx, model_name in enumerate(models):
    row = model_idx // n_cols + 1
    col = model_idx % n_cols + 1

    if model_name not in results:
        continue

    # Add reference arrow at 0 degrees
    fig.add_trace(
        go.Scatterpolar(
            r=[1.02],
            theta=[0],
            name="feature direction",
            marker=dict(size=20, symbol="arrow-right", color="black"),
            mode="markers",
            showlegend=model_idx == 0,  # Only show in legend for first model
        ),
        row=row,
        col=col,
    )

    # Track which tasks we've displayed for this model
    task_color_index = 0

    # Sort the tasks for consistent color assignment
    sorted_tasks = sorted(results[model_name].keys())
    
    for task_display_name in sorted_tasks:
        task_color = colors[task_color_index % len(colors)]
        task_color_index += 1

        # Prepare angle and score data
        angles = sorted(results[model_name][task_display_name].keys())
        scores = [results[model_name][task_display_name][angle] for angle in angles]

        # Close the loop for a complete polar plot
        angles_closed = np.append(angles, angles[0])
        scores_closed = np.append(scores, scores[0])

        # Plot the task scores with solid lines
        fig.add_trace(
            go.Scatterpolar(
                r=scores_closed,
                theta=angles_closed,
                name=task_display_name,
                line=dict(width=2, color=task_color),
                mode="lines",
                showlegend=model_idx == 0,  # Only show in legend for first model
            ),
            row=row,
            col=col,
        )
        
        # Add baseline as dashed circular line if available
        if model_name in baselines and task_display_name in baselines[model_name]:
            baseline_score = baselines[model_name][task_display_name]
            baseline_theta = np.linspace(0, 360, 361)

            # Use dash line for baselines but don't add to legend
            fig.add_trace(
                go.Scatterpolar(
                    r=[baseline_score] * len(baseline_theta),
                    theta=baseline_theta,
                    line=dict(width=1.5, color=task_color, dash="dot"),
                    mode="lines",
                    opacity=0.5,
                    showlegend=False,  # Don't add individual baselines to legend
                ),
                row=row,
                col=col,
            )
    print([(model_name, task, np.min(list(results[model_name][task].values()))) for task in results[model_name].keys() for task in sorted_tasks])
    # print(model_name,baselines[model_name])

# Update layout for each subplot
for i in range(1, n_models + 1):
    row = (i - 1) // n_cols + 1
    col = (i - 1) % n_cols + 1

    # Get the corresponding polar subplot key
    polar_key = f'polar{i if i > 1 else ""}'

    # Set consistent range and angle markings across plots
    fig.update_layout(
        {
            polar_key: dict(
                radialaxis=dict(
                    visible=True,
                    range=[0.0, 1.02],
                    dtick=0.2,
                    # tickvals=[0.3, 0.5, 0.7, 0.9, 1],
                    tickfont=dict(size=22),
                ),
                angularaxis=dict(
                    # direction="clockwise",
                    # rotation=0,  # 0 degrees at the right (East)
                    # period=360,
                    # tickmode="array",
                    # tickvals=list(
                    #     range(0, 360, 10)
                    # ),  # Show all degrees in 10 intervals
                    # ticktext=[
                    #     f"{i}°" for i in range(0, 360, 10)
                    # ],  # Show all tick labels
                    tickfont=dict(size=16),  # Smaller font to fit all labels
                    dtick=10,
                ),
            )
        }
    )

# Global layout settings
fig.update_layout(
    height=2000,
    width=1200,
    # title_text="Angular Steering Effects on Model Performance",
    showlegend=True,
    legend=dict(
        orientation="h",
        # yanchor="top",
        y=0.0,
        xanchor="center",
        x=0.5,
        # entrywidth=0,
        font=dict(size=30),
    ),
    margin=dict(l=50, r=50, t=20, b=0),
)
fig.update_annotations(font=dict(size=36), yshift=-20)

fig.show()

fig.write_image(
    visualization_dir / "PID_eval_adaptive-tinyBenchmark-all_models-vertical.pdf",
    width=1200,
    height=2000,
    scale=5,
)

## Perplexity scores


In [ ]:
import json
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path
import glob


def filter_perplexity(perplexity_data, threshold=100):
    """
    Filter perplexity data to remove outliers.
    E.g. in `qwen2.5-3B/rotated/max_sim-pca/290/34` there is a perplexity of 2.8e6
    because the generation is empty
    """
    return [p for p in perplexity_data if p < threshold]


MODELS = [
    "Qwen/Qwen2.5-3B-Instruct",
    "Qwen/Qwen2.5-7B-Instruct",
    "Qwen/Qwen2.5-14B-Instruct",
    "meta-llama/Llama-3.2-3B-Instruct",
    "meta-llama/Llama-3.1-8B-Instruct",
    "google/gemma-2-9b-it",
]

# Base output directory
output_dir = Path("output")

generation_only = False

# Find all perplexity evaluation files
if generation_only:
    perplexity_files = glob.glob(
        str(output_dir / "*" / "eval-perplexity-generation_only.json")
    )
else:
    perplexity_files = glob.glob(str(output_dir / "*" / "eval-perplexity.json"))

# Create figure with subplots based on number of models
n_models = len(MODELS)
n_cols = min(2, n_models)
n_rows = (n_models + n_cols - 1) // n_cols
# n_cols = 1
# n_rows = n_models

# Colors for different modes
colors = px.colors.qualitative.Plotly

# Calculate model-specific max perplexity values
all_perplexity_data = {}

for perplexity_file in perplexity_files:
    model_name = Path(perplexity_file).parent.name
    with open(perplexity_file, "r") as f:
        data = json.load(f)
        all_perplexity_data[model_name] = data

        # Calculate max perplexity for this specific model
        model_max = 0

        # Check baseline max
        if "harmful-en-baseline" in data:
            model_max = max(model_max, max(data["harmful-en-baseline"]))

if n_models > 1:
    from plotly.subplots import make_subplots

    fig = make_subplots(
        rows=n_rows,
        cols=n_cols,
        specs=[[{"type": "polar"} for _ in range(n_cols)] for _ in range(n_rows)],
        subplot_titles=[m.split('/')[-1] for m in MODELS],
        vertical_spacing=0.01,
        horizontal_spacing=0.1,
    )
else:
    fig = go.Figure()

# Add reference baseline trace only once for legend
fig.add_trace(
    go.Scatterpolar(
        r=[0],
        theta=[0],
        name="no steering",
        line=dict(width=2, color="black", dash="dot"),
        mode="lines",
        opacity=0.5,
        showlegend=True,
    ),
    row=1 if n_models > 1 else None,
    col=1 if n_models > 1 else None,
)

# Process each perplexity file
for model_idx, model_id in enumerate(MODELS):
    chosen_direction = DIR_ID_MAP[model_id]
    print(model_id)
    # print(chosen_direction)


    model_name = model_id.split("/")[-1]
    perplexity_data = all_perplexity_data[model_name]

    row = model_idx // n_cols + 1 if n_models > 1 else None
    col = model_idx % n_cols + 1 if n_models > 1 else None

    # Extract modes (rotated, adaptive, etc.)
    modes = []
    for key in perplexity_data.keys():
        if chosen_direction not in key:
            continue
        print(key)
        if key.startswith("harmful-en-dir") and not key.endswith("baseline"):
            mode = "non-adaptive" if "rotated" in key else "adaptive"
            modes.append((key, mode))
    modes.sort()

    # Process each mode
    for mode_idx, (mode_key, mode_name) in enumerate(modes):
        angles = []
        mean_perplexities = []

        # Calculate mean perplexity for each angle
        for angle_str, perplexities in perplexity_data[mode_key].items():
            if angle_str.isdigit() and perplexities:
                perplexities = filter_perplexity(perplexities)
                angle = int(angle_str)
                mean_perp = np.mean(perplexities)
                angles.append(angle)
                mean_perplexities.append(mean_perp)

        # Sort by angle
        sorted_indices = np.argsort(angles)
        angles = np.array(angles)[sorted_indices]
        mean_perplexities = np.array(mean_perplexities)[sorted_indices]

        # Add one more point to close the loop
        if len(angles) > 1:
            angles_closed = np.append(angles, angles[0])
            perplexities_closed = np.append(mean_perplexities, mean_perplexities[0])

            # Add to plot
            fig.add_trace(
                go.Scatterpolar(
                    r=perplexities_closed,
                    theta=angles_closed,
                    name=f"{mode_name}",
                    line=dict(width=2, color=colors[mode_idx % len(colors)]),
                    mode="lines",
                    showlegend=model_idx == 0,  # Only show in legend for first model
                ),
                row=row,
                col=col,
            )

    max_r = max(mean_perplexities)

    # Add reference star at 0 degrees
    fig.add_trace(
        go.Scatterpolar(
            r=[max_r * 1.02],
            theta=[0],
            name="feature direction",
            marker=dict(size=20, symbol="arrow-right", color="black"),
            mode="markers",
            showlegend=model_idx == 0,  # Only show in legend for first model
            legend="legend",
        ),
        row=row,
        col=col,
    )

    # Add baseline if available
    if "harmful-en-baseline" in perplexity_data:
        baseline_perplexity = np.mean(perplexity_data["harmful-en-baseline"])
        max_r = max(max_r, baseline_perplexity)
        baseline_theta = np.linspace(0, 360, 100)

        fig.add_trace(
            go.Scatterpolar(
                r=[baseline_perplexity] * len(baseline_theta),
                theta=baseline_theta,
                name="Baseline",
                line=dict(width=1.5, color="black", dash="dot"),
                mode="lines",
                opacity=0.5,
                showlegend=False,  # Don't add individual baselines to legend
                legend="legend",
            ),
            row=row,
            col=col,
        )


# Update layout with model-specific ranges
if n_models > 1:
    for i in range(1, n_models + 1):
        row = (i - 1) // n_cols + 1
        col = (i - 1) % n_cols + 1
        polar_key = f'polar{i if i > 1 else ""}'

        fig.update_layout(
            {
                polar_key: dict(
                    radialaxis=dict(
                        visible=True,
                        # range=[0.3, 1.02],
                        # dtick=0.2,
                        # tickvals=[0.3, 0.5, 0.7, 0.9, 1],
                        tickfont=dict(size=22),
                    ),
                    angularaxis=dict(
                        tickvals=list(range(0, 360, 10)),
                        ticktext=[f"{i}°" for i in range(0, 360, 10)],
                        tickfont=dict(size=16),
                    ),
                )
            }
        )
else:
    fig.update_layout(
        polar=dict(
            radialaxis=dict(visible=True, title="Average Perplexity"),
            angularaxis=dict(
                tickvals=list(range(0, 360, 10)),
                ticktext=[f"{i}°" for i in range(0, 360, 10)],
                tickfont=dict(size=16),
            ),
        ),
    )

fig.update_layout(
    height=2000,
    width=1200,
    # title_text="Angular Steering Effects on Model Perplexity",
    showlegend=True,
    # legend=dict(
    #     orientation="h",
    #     yanchor="bottom",
    #     y=-0.1 if n_models > 1 else -0.2,
    #     xanchor="center",
    #     x=0.5,
    # ),
    legend=dict(
        orientation="h",
        # yanchor="top",
        y=0.0,
        xanchor="center",
        x=0.5,
        font=dict(size=30),
        # entrywidth=100,
        entrywidthmode="pixels",
    ),
    # legend2=dict(
    #     orientation="h",
    #     # yanchor="top",
    #     y=-0.01,
    #     xanchor="center",
    #     x=0.5,
    #     font=dict(size=16),
    # ),
    margin=dict(l=50, r=50, t=20, b=140),
)
fig.update_annotations(font=dict(size=36), yshift=-20)

fig.show()

# save figure
if generation_only:
    output_name = "eval-ppl-generation_only"
else:
    output_name = "eval-ppl"
fig.write_image(
    visualization_dir / f"{output_name}-vertical.pdf",
    height=2000,
    width=1200,
    scale=5,
)

## Geometric interpretation of Angular Steering


In [ ]:
import plotly
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# Define base vectors
d_1stPC_3d = np.array([1.5, 0.0, 0.5])
d_3d = np.array([0.5, 1.0, 0.0])
d_3d = d_3d / np.linalg.norm(d_3d)

alpha = 1.0
h_add_3d = d_1stPC_3d + alpha * d_3d
d_ortho_3d = d_1stPC_3d - np.dot(d_1stPC_3d, d_3d) * d_3d

# Normalize versions
h_norm = d_1stPC_3d / np.linalg.norm(d_1stPC_3d)
h_add_norm = h_add_3d / np.linalg.norm(h_add_3d)
h_ablate_norm = d_ortho_3d / np.linalg.norm(d_ortho_3d)

# Plane that spans h and d
extended_h = 1.2 * d_1stPC_3d
extended_d = 1.2 * d_3d
origin = np.array([0, 0, 0])
corner1 = 1.2 * extended_h + 1.4 * extended_d
corner2 = 1.2 * extended_h - 0.8 * extended_d
corner3 = 0.5 * -extended_h - 0.8 * extended_d
corner4 = 0.5 * -extended_h + 1.4 * extended_d
plane_vertices = [corner1, corner2, corner3, corner4]

# Create figure and axes
fig = plt.figure(figsize=(14, 6))
ax1 = fig.add_subplot(121, projection="3d")
ax2 = fig.add_subplot(122, projection="3d")


# Right angle marker
def draw_right_angle_marker(ax, vec1, vec2, scale=0.2):
    v1 = vec1 / np.linalg.norm(vec1)
    v2 = vec2 / np.linalg.norm(vec2)
    base = np.array([0, 0, 0])
    corner = base + scale * v1
    offset1 = corner + scale * v2
    offset2 = offset1 - scale * v1
    ax.plot(
        [corner[0], offset1[0]],
        [corner[1], offset1[1]],
        [corner[2], offset1[2]],
        color="gray",
    )
    ax.plot(
        [offset1[0], offset2[0]],
        [offset1[1], offset2[1]],
        [offset1[2], offset2[2]],
        color="gray",
    )


# Rotation arc
def draw_arc(ax, v_from, v_to, color):
    v_from = v_from / np.linalg.norm(v_from)
    v_to = v_to / np.linalg.norm(v_to)
    theta = np.linspace(0, np.arccos(np.clip(np.dot(v_from, v_to), -1, 1)), 50)
    axis = np.cross(v_from, v_to)
    axis = axis / np.linalg.norm(axis)
    arc = np.array(
        [1.0 * (np.cos(t) * v_from + np.sin(t) * np.cross(axis, v_from)) for t in theta]
    )
    ax.plot(arc[:, 0], arc[:, 1], arc[:, 2], color=color, linestyle="--")


def draw_vector_arrow(ax, vec, color, label):
    ax.plot([0, vec[0]], [0, vec[1]], [0, vec[2]], color=color, label=label)
    direction = vec / np.linalg.norm(vec)
    side = np.cross(direction, np.array([0, 0, 1]))
    if np.linalg.norm(side) < 1e-6:
        side = np.cross(direction, np.array([0, 1, 0]))
    side = side / np.linalg.norm(side) * 0.05
    head_length = 0.1
    tip = vec
    left = tip - head_length * direction + side
    right = tip - head_length * direction - side
    ax.plot([tip[0], left[0]], [tip[1], left[1]], [tip[2], left[2]], color=color)
    ax.plot([tip[0], right[0]], [tip[1], right[1]], [tip[2], right[2]], color=color)


# General plotting function
def plot_vectors(
    ax,
    h_vec,
    h_add_vec,
    h_ablate_vec,
    title,
    show_addition_lines=False,
    show_right_angle=False,
    show_rotation_arc=True,
    show_legend=True,
):

    draw_vector_arrow(
        ax,
        h_vec,
        color=plotly.colors.qualitative.Plotly[0],
        label="$\\mathbf{h}$ (activation)",
    )
    draw_vector_arrow(
        ax,
        d_3d,
        color=plotly.colors.qualitative.Plotly[1],
        label="$\\mathbf{d}_\\text{feature}$ (feature direction)",
    )
    draw_vector_arrow(
        ax,
        h_add_vec,
        color=plotly.colors.qualitative.Plotly[3],
        label=(
            "$\\mathbf{h} + \\alpha \\mathbf{d}_\\text{feature}$ (activation addition,"
            " $\\alpha=1$)"
        ),
    )
    draw_vector_arrow(
        ax,
        h_ablate_vec,
        color=plotly.colors.qualitative.Plotly[2],
        label="$\\mathbf{h}_\\perp$ (directional ablation)",
    )

    ax.add_collection3d(Poly3DCollection([plane_vertices], color="gray", alpha=0.1))

    if show_addition_lines:
        ax.plot(
            [h_vec[0], h_add_vec[0]],
            [h_vec[1], h_add_vec[1]],
            [h_vec[2], h_add_vec[2]],
            linestyle="dashed",
            color="gray",
            linewidth=1,
        )
        ax.plot(
            [d_3d[0], h_add_vec[0]],
            [d_3d[1], h_add_vec[1]],
            [d_3d[2], h_add_vec[2]],
            linestyle="dashed",
            color="gray",
            linewidth=1,
        )

    if show_right_angle:
        draw_right_angle_marker(ax, d_3d, h_ablate_vec)

    if show_rotation_arc:
        draw_arc(ax, d_3d, h_ablate_vec, (0, 0, 1, 0.3))
        # draw_arc(ax, h_vec, h_add_vec, "orange")
        # draw_arc(ax, h_vec, h_ablate_vec, "red")

    ax.set_title(title, fontsize=20)
    ax.set_xlim([-1, 2])
    ax.set_ylim([-1, 2])
    ax.set_zlim([-1, 2])
    ax.view_init(elev=30, azim=135)
    ax.grid(True)
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.set_zticklabels([])
    ax.xaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))
    ax.yaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))
    ax.zaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))

    if show_legend:
        ax.legend(fontsize=18, loc="upper left", bbox_to_anchor=(0, 0.3))


# Plot left: before normalization
plot_vectors(
    ax1,
    d_1stPC_3d,
    h_add_3d,
    d_ortho_3d,
    "Before Normalization",
    show_addition_lines=True,
    show_right_angle=True,
    show_rotation_arc=False,
    show_legend=False,
)

# Plot right: after normalization
plot_vectors(
    ax2,
    h_norm,
    h_add_norm,
    h_ablate_norm,
    "After Normalization",
    show_addition_lines=False,
    show_right_angle=True,
    show_rotation_arc=True,
    show_legend=True,
)

# Connect the two views
fig.text(0.51, 0.5, "➜", fontsize=20, ha="center", va="center")
# plt.tight_layout()
fig.subplots_adjust(left=0.0, right=1.0, top=1.0, bottom=0.0, wspace=-0.2)

plt.show()

fig.savefig("visualization/steering_methods.pdf", dpi=300, bbox_inches="tight")

## Rotation within a steering plane (wip)


In [ ]:
import plotly
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# Define base vectors
h_3d = np.array([1, 0.5, 1])
d_1stPC_3d = np.array([1.0, -0.5, 0.0])
d_3d = np.array([0.0, 1.0, 0.0])
d_3d = d_3d / np.linalg.norm(d_3d)

alpha = 1.0
h_add_3d = d_1stPC_3d + alpha * d_3d
d_ortho_3d = d_1stPC_3d - np.dot(d_1stPC_3d, d_3d) * d_3d

# Normalize versions
h_norm = d_1stPC_3d / np.linalg.norm(d_1stPC_3d)
h_add_norm = h_add_3d / np.linalg.norm(h_add_3d)
h_ablate_norm = d_ortho_3d / np.linalg.norm(d_ortho_3d)

# Plane that spans h and d
extended_h = 1.2 * d_ortho_3d
extended_d = 1.2 * d_3d
origin = np.array([0, 0, 0])
corner1 = 1.2 * extended_h + 1.4 * extended_d
corner2 = 1.2 * extended_h - 0.8 * extended_d
corner3 = 0.5 * -extended_h - 0.8 * extended_d
corner4 = 0.5 * -extended_h + 1.4 * extended_d
plane_vertices = [corner1, corner2, corner3, corner4]

# Create figure and axes
fig = plt.figure(figsize=(14, 6))
ax1 = fig.add_subplot(121, projection="3d")
# ax2 = fig.add_subplot(122, projection="3d")


# Right angle marker
def draw_right_angle_marker(ax, vec1, vec2, scale=0.2):
    v1 = vec1 / np.linalg.norm(vec1)
    v2 = vec2 / np.linalg.norm(vec2)
    base = np.array([0, 0, 0])
    corner = base + scale * v1
    offset1 = corner + scale * v2
    offset2 = offset1 - scale * v1
    ax.plot(
        [corner[0], offset1[0]],
        [corner[1], offset1[1]],
        [corner[2], offset1[2]],
        color="gray",
    )
    ax.plot(
        [offset1[0], offset2[0]],
        [offset1[1], offset2[1]],
        [offset1[2], offset2[2]],
        color="gray",
    )


# Rotation arc
def draw_arc(ax, v_from, v_to, color):
    v_from = v_from / np.linalg.norm(v_from)
    v_to = v_to / np.linalg.norm(v_to)
    theta = np.linspace(0, np.arccos(np.clip(np.dot(v_from, v_to), -1, 1)), 50)
    axis = np.cross(v_from, v_to)
    axis = axis / np.linalg.norm(axis)
    arc = np.array(
        [1.0 * (np.cos(t) * v_from + np.sin(t) * np.cross(axis, v_from)) for t in theta]
    )
    ax.plot(arc[:, 0], arc[:, 1], arc[:, 2], color=color, linestyle="--")


def draw_vector_arrow(ax, vec, color, label):
    ax.plot([0, vec[0]], [0, vec[1]], [0, vec[2]], color=color, label=label)
    direction = vec / np.linalg.norm(vec)
    side = np.cross(direction, np.array([0, 0, 1]))
    if np.linalg.norm(side) < 1e-6:
        side = np.cross(direction, np.array([0, 1, 0]))
    side = side / np.linalg.norm(side) * 0.05
    head_length = 0.1
    tip = vec
    left = tip - head_length * direction + side
    right = tip - head_length * direction - side
    ax.plot([tip[0], left[0]], [tip[1], left[1]], [tip[2], left[2]], color=color)
    ax.plot([tip[0], right[0]], [tip[1], right[1]], [tip[2], right[2]], color=color)


# General plotting function

title = "Angular Steering"
show_addition_lines = (True,)
show_right_angle = (True,)
show_rotation_arc = (False,)
show_legend = (False,)

draw_vector_arrow(
    ax1,
    h_3d,
    color=plotly.colors.qualitative.Plotly[0],
    label="$\\mathbf{h}$ (activation)",
)

draw_vector_arrow(
    ax1,
    d_3d,
    color=plotly.colors.qualitative.Plotly[1],
    label="$\\mathbf{d}_\\text{feature}$ (feature direction)",
)

draw_vector_arrow(
    ax1,
    d_1stPC_3d,
    color=plotly.colors.qualitative.Plotly[6],
    label="$\mathbf{d}_\\text{1stPC}$",
)

ax1.add_collection3d(Poly3DCollection([plane_vertices], color="gray", alpha=0.1))


if show_right_angle:
    draw_right_angle_marker(ax1, d_3d, d_1stPC_3d)

if show_rotation_arc:
    draw_arc(ax1, d_3d, d_1stPC_3d, (0, 0, 1, 0.3))
    # draw_arc(ax, h_vec, h_add_vec, "orange")
    # draw_arc(ax, h_vec, h_ablate_vec, "red")

ax1.set_title(title, fontsize=20)
ax1.set_xlim([-1, 2])
ax1.set_ylim([-1, 2])
ax1.set_zlim([-1, 2])
ax1.view_init(elev=30, azim=135)
ax1.grid(True)
ax1.set_xticklabels([])
ax1.set_yticklabels([])
ax1.set_zticklabels([])
ax1.xaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))
ax1.yaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))
ax1.zaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))

if show_legend:
    ax1.legend(fontsize=18, loc="upper left", bbox_to_anchor=(0, 0.3))


# Connect the two views
# fig.text(0.51, 0.5, "➜", fontsize=20, ha="center", va="center")
# plt.tight_layout()
fig.subplots_adjust(left=0.0, right=1.0, top=1.0, bottom=0.0, wspace=-0.2)

plt.show()

fig.savefig("visualization/steering_methods.png", dpi=300, bbox_inches="tight")